In [12]:
import pandas as pd
import json
import numpy as np

kyr_data = []
with open("../kyr_wsd_dataset/validation_set.json") as f:
    kyr_data = json.load(f)

kyr_df = pd.DataFrame(kyr_data)

In [14]:
kyr_sentences = []
with open("../auxiliary_data/apertium_constitution.json") as f:
    kyr_sentences = json.load(f)

from streamparser import LexicalUnit

# get LexicalUnit from string
def get_lex_units(units):
    return [LexicalUnit(unit) for unit in units]

kyr_units = []
for sent in kyr_sentences:
    processed_sent = get_lex_units(sent)
    kyr_units.append(processed_sent)

# few cases where apertium finds multiple lemmas, wrong cases are filtered manually
hard_dct = {
    frozenset(('ата', 'атан')): 'ата',
    frozenset(('бал', 'бала')): 'бала',
    frozenset(('жара', 'жаран')): 'жаран',
    frozenset(('күн', 'күнү')): 'күн',
    frozenset(('уй', 'уюм')): 'уюм'
}

def get_lemma(sentence, kyr_token_id):
    target_unit = sentence[kyr_token_id]
    nouns = set()

    # iterate over apertiums analysis and extract all lemmas
    for reading in target_unit.readings:
        for sread in reading:
            if 'n' in sread.tags:
                # extracting baseform of kyrgyz word
                nouns.add(sread.baseform)

    # if there is only one lemma everything is ok
    if len(nouns) == 1:
        lemma = nouns.pop()
    else:
        # few lemmas - one of our hard cases
        lemma = hard_dct[frozenset(nouns)]

    return lemma

In [ ]:
grouped = kyr_df.groupby('instance_id')
kyr_gold = []
kyr_queries = []
for instance_id, group in grouped:
    context_tgt = group['context'].tolist()[0] # all contexts are the same in one group
    context_tgt_split = context_tgt.split("[TGT]")
    context = " <WSD> ".join(context_tgt_split)
    labels = group['label'].tolist()

    sentence_id, kyr_token_id = [int(a) for a in instance_id.split("_")]
    # kyr_units is kyrgyz senteces as an apertium parsing result
    wsd_word = get_lemma(kyr_units[sentence_id], kyr_token_id)

    glosses = group['gloss'].tolist()
    descriptions = [f"{i+1}) {gloss}" for i, gloss in enumerate(glosses)]

    # forming gold dataset
    kyr_gold.append(
        {
            "item_index": instance_id, 
            "word": wsd_word,
            "selected_option": int(np.argmax(labels))
        }
    )
    
    # forming dict for API call to deepseek
    kyr_queries.append(
        {
            "sentence_id": sentence_id,
            "kyr_token_id": kyr_token_id,
            "sentence": context,
            "word": wsd_word,
            "descriptions": descriptions
        }
    )

In [39]:
kyr_gold_df = pd.DataFrame(kyr_gold)
kyr_gold_df.to_csv("../gold_data/kyr_gold.csv", index=False)

In [40]:
kyr_queries[1]

{'sentence_id': 0,
 'kyr_token_id': 34,
 'sentence': 'Биз , Кыргыз Республикасынын эли , өз тагдырыбызды өз алдынча чечүү укугун туу тутуп ; укук үстөмдүүлүгүн , адилеттүүлүктү жана тең укуктуулукту камсыз кылуу максатында ; чыныгы элдик бийликтин негиздерин бекемдеп ; ата - бабабыздын наркын ,  <WSD>  салтын  <WSD>  сактап , Айкөл Манастын осуяттарына таянып , биримдикте , тынчтыкта жана ынтымакта табият менен таттуу мамиледе жашоону улап ; Кыргыз Республикасынын элинин укуктарын жана кызыкчылыктарын коргоп ; мамлекеттүүлүктү сактоо жана чыңдоо үчүн майтарылбас эркти көрсөтүп ; адамдын жана жарандын укуктарын жана эркиндиктерин коргоого жана урматтоого умтулууну ырастап ; жалпы адамзаттык принциптерди жана баалуулуктарды таанып ; социалдык адилеттикке , экономикалык бакубатчылыкка , билимге , илимге жана руханий өнүгүүгө умтулуп ; элдин эркиндиги үчүн курман болгон баатырларды эсте сактап ; азыркы жана келечек муундардын алдында Мекен үчүн жоопкерчиликти түшүнүп , ушул Конституцияны к

In [50]:
def get_cot_prompt(word: str, sentence: str, synset_descriptions):
    parts = ['You are going to identify the corresponding sense tag of an ambiguity word in Kyrgyz sentences. Do the following tasks.', 
        f'1. {word} has different meanings. Below are possible meanings. Comprehend the sensetags and meanings.']
    parts.extend(synset_descriptions)
    parts.append('2. Now examine the sentence below. You are going to identify the most suitable meaning for ambiguity word.')
    parts.append(sentence)
    parts.extend([
        '3. Try to identify the meaning of the word in the above sentence which is enclosed with the <WSD>. You can think of the real meaning of \
sentence and decide the most suitable meaning for the word.',
        '4. Based on the identified meaning, try to find the most appropriate sense from the below. You are given definition of each sense tag too.'
    ])
    parts.extend(synset_descriptions)
    parts.extend([
        '5. If you have more than one senses identified after above steps, you can return the numbers in order of confidence level.',
        '6. Return JSON object that contains the ambiguity word and the finalized senseIDs. Use the following format for the output.',
        '<JSON Object with fields ambiguity_word and senseIDs >'
    ])
    return "\n".join(parts)

In [51]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
key = os.getenv("DEEPSEEK")

client = OpenAI(
    api_key=key,
    base_url="https://api.deepseek.com",
)

In [68]:
response.choices[0].message.content

''

In [69]:
import tqdm

answers = []
for query in tqdm.tqdm(kyr_queries):
    prompt = get_cot_prompt(query['word'], query['sentence'], query['descriptions'])
    messages = [{"role": "system", "content": prompt}]
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=messages,
        response_format={
            'type': 'json_object'
        }
    )
    answers.append(response.choices[0].message.content)


100%|██████████| 438/438 [22:57<00:00,  3.15s/it]


In [77]:
answers

['{\n  "ambiguity_word": "баатыр",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "салт",\n  "senseIDs": [1]\n}',
 '{"ambiguity_word": "илим", "senseIDs": [1]}',
 '{"ambiguity_word":"сот","senseIDs":[1]}',
 '{\n  "ambiguity_word": "мыйзам",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "мазмун",\n  "senseIDs": [1]\n}',
 '{"ambiguity_word": "мыйзам", "senseIDs": [1]}',
 '{\n  "ambiguity_word": "мыйзам",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "мыйзам",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "аймагынын",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "абалы",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "тили",\n  "senseIDs": [3]\n}',
 '{\n  "ambiguity_word": "мыйзам",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "мыйзамда",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "топ",\n  "senseIDs": [3]\n}',
 '{\n  "ambiguity_word": "мыйзам",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "сот",\n  "senseIDs": [1]\n}',
 '{\n  "ambiguity_word": "аялдар",\n  

In [80]:
json.loads(answers[0])

{'ambiguity_word': 'баатыр', 'senseIDs': [1]}

In [86]:
for query, answer in tqdm.tqdm(zip(kyr_queries, answers)):
    try:
        answer_json = json.loads(answer)
        query["senseIDs"] = answer_json["senseIDs"]
    except Exception as e:
        prompt = get_cot_prompt(query['word'], query['sentence'], query['descriptions'])
        messages = [{"role": "system", "content": prompt}]
        response = client.chat.completions.create(
            model="deepseek-reasoner",
            messages=messages,
            response_format={
                'type': 'json_object'
            }
        )
        sense_ids = json.loads(response.choices[0].message.content)
        query["senseIDs"] = sense_ids["senseIDs"]

438it [00:04, 90.82it/s]


In [ ]:
for query in kyr_queries:
    query[]
    if "annotation" in query:
        query.pop("annotation")

In [90]:
for query in kyr_queries:
    sense_ids = query['senseIDs']
    query['senseIDs'] = [int(id) - 1 for id in sense_ids]

In [91]:
kyr_queries

[{'sentence_id': 0,
  'kyr_token_id': 108,
  'sentence': 'Биз , Кыргыз Республикасынын эли , өз тагдырыбызды өз алдынча чечүү укугун туу тутуп ; укук үстөмдүүлүгүн , адилеттүүлүктү жана тең укуктуулукту камсыз кылуу максатында ; чыныгы элдик бийликтин негиздерин бекемдеп ; ата - бабабыздын наркын , салтын сактап , Айкөл Манастын осуяттарына таянып , биримдикте , тынчтыкта жана ынтымакта табият менен таттуу мамиледе жашоону улап ; Кыргыз Республикасынын элинин укуктарын жана кызыкчылыктарын коргоп ; мамлекеттүүлүктү сактоо жана чыңдоо үчүн майтарылбас эркти көрсөтүп ; адамдын жана жарандын укуктарын жана эркиндиктерин коргоого жана урматтоого умтулууну ырастап ; жалпы адамзаттык принциптерди жана баалуулуктарды таанып ; социалдык адилеттикке , экономикалык бакубатчылыкка , билимге , илимге жана руханий өнүгүүгө умтулуп ; элдин эркиндиги үчүн курман болгон  <WSD>  баатырларды  <WSD>  эсте сактап ; азыркы жана келечек муундардын алдында Мекен үчүн жоопкерчиликти түшүнүп , ушул Конституция

In [93]:
with open('../deepseek_answers/deepseek_kyr_results.json', 'w', encoding="utf-8") as f:
    json.dump(kyr_queries, f, indent=4, ensure_ascii=False)